<div style="background: linear-gradient(135deg, #000000 0%, #1a1a1a 100%); padding: 40px; border-radius: 15px; border-left: 5px solid #FF6B35; margin: 20px 0;">
    <h1 style="color: #FF6B35; font-family: 'Segoe UI', Arial, sans-serif; font-weight: 700; margin: 0; font-size: 2.5em;">
        🎓 Práctica 3
    </h1>
    <div style="height: 3px; background: linear-gradient(90deg, #FF6B35, #FF8C42); margin: 20px 0; border-radius: 2px;"></div>
    
<div style="margin-top: 30px; padding: 15px; background: rgba(255, 107, 53, 0.1); border-radius: 10px; border: 1px solid #FF6B35;">
    <p style="color: #ffffff; font-size: 1.3em; margin: 0; font-weight: 500;">WSD Word Sense Disambiguation</p>
</div>

<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 30px; margin-top: 25px;">
<div>
    <p style="color: #FF8C42; font-size: 0.9em; margin: 5px 0; font-weight: 600;">ESTUDIANTES</p>
    <p style="color: #ffffff; font-size: 1.2em; margin: 5px 0;">Bustillos Cruz Jonatan</p>
    <p style="color: #ffffff; font-size: 1.2em; margin: 5px 0;">Gutiérrez Sánchez Angélica</p>

<p style="color: #FF8C42; font-size: 0.9em; margin: 15px 0 5px 0; font-weight: 600;">MATERIA</p>
<p style="color: #ffffff; font-size: 1.2em; margin: 5px 0;">Aplicaciones de Lenguaje Natural</p>
</div>
<div>
    <p style="color: #FF8C42; font-size: 0.9em; margin: 5px 0; font-weight: 600;">PROFESORA</p>
    <p style="color: #ffffff; font-size: 1.2em; margin: 5px 0;">Elizabeth Moreno Galván</p>
    
<p style="color: #FF8C42; font-size: 0.9em; margin: 15px 0 5px 0; font-weight: 600;">FECHA</p>
<p style="color: #ffffff; font-size: 1.2em; margin: 5px 0;">Septiembre 2025</p>

<p style="color: #FF8C42; font-size: 0.9em; margin: 15px 0 5px 0; font-weight: 600;">GRUPO</p>
<p style="color: #ffffff; font-size: 1.2em; margin: 5px 0;">6BM1</p>
</div>
</div>


</div>

## 📋 ÍNDICE NAVEGABLE

### 📑 Secciones Principales
1. **[Instrucciones](#def_problema)** - Definición de instrucciones
1. **[Imports y Configuración](#b_1)** - Librerías y modelos base
2. **[Diccionario Personalizado](#b_2)** - Definiciones y sentidos de palabras
3. **[Tokenización Mejorada](#b_3)** - Preprocesamiento con lematización
4. **[Algoritmo Weighted Lesk](#b_4)** - Desambiguación léxica
5. **[Traducción con Desambiguación](#b_5)** - Integración Lesk + Transformer
6. **[Ejecución del Encargo](#b_6)** - Caso de estudio y pruebas
7. **[Análisis Comparativo](#b_7)** - Evaluación de resultados
7. **[Conclusiones](#conclusiones)** - Aprendizajes y recomendaciones

<a id="def_problema"></a>
## 🎯 1. Instrucciones

<div style="background: rgba(255, 107, 53, 0.1); padding: 20px; border-radius: 10px; border-left: 4px solid #FF6B35; margin: 15px 0;">
<p style="color: #ffffff; font-size: 1.1em; line-height: 1.6;">
    \Instrucciones:

Tomando como ejemplo la frase: "Fui por dinero, un préstamo e hice una transacción en el banco"

Desambigua utilizando el método <strong style="color: #FF6B35;">LESK</strong> la palabra "banco" de forma que puedas traducir lo más fiel posible la frase: "I went to the bank for money, a loan, and made a transaction".

La traducción debe ser única de ésa frase, no se busca que se cree un traductor general, puede utilizarse una librería para traducción, la idea es que la palabra banco sea traducida como "bank" y no como "bench".
</p>
</div>


<a id="b_1"></a>
# BLOQUE 1: Imports y Configuración Inicial


In [ ]:
# Imports necesarios
import nltk
from nltk.corpus import stopwords
import spacy
from collections import Counter
import numpy as np
from transformers import pipeline
import json
from typing import List, Tuple, Set, Dict
import warnings
warnings.filterwarnings('ignore')

# Descargas necesarias (ejecutar solo la primera vez)
# nltk.download('stopwords')

# Cargar modelo de Spacy para español
try:
    NLP_ES = spacy.load("es_core_news_sm")
    print("✅ Modelo Spacy cargado correctamente")
except:
    print("⚠️ Instalar: python -m spacy download es_core_news_sm")
    NLP_ES = None

# Configurar stopwords
STOPWORDS_ES = set(stopwords.words('spanish'))

# Cargar traductor (se descargará la primera vez)
print("🔄 Cargando modelo de traducción...")
try:
    TRANSLATOR = pipeline(
        "translation",
        model="Helsinki-NLP/opus-mt-es-en",
        device=-1  # CPU
    )
    print("✅ Traductor cargado correctamente")
except Exception as e:
    print(f"❌ Error al cargar traductor: {e}")
    TRANSLATOR = None

c:\Users\Joni\.conda\envs\noticias-nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Modelo Spacy cargado correctamente
🔄 Cargando modelo de traducción...


Device set to use cpu


✅ Traductor cargado correctamente


<a id="b_2"></a>
#  BLOQUE 2: Diccionario Personalizado


In [ ]:
# Diccionario personalizado con definiciones de calidad
DICCIONARIO_ESPANOL = {
    "banco": [
        {
            "id": "banco_asiento",
            "definicion": "Asiento largo con o sin respaldo, generalmente de madera o metal, donde pueden sentarse varias personas",
            "ejemplos": [
                "Me senté en el banco del parque a descansar",
                "El banco de la plaza estaba ocupado por estudiantes",
                "Colocaron bancos nuevos en el jardín público"
            ],
            "sinonimos": ["asiento", "banca", "escaño"],
            "categoria": "sustantivo",
            "traduccion_en": "bench"
        },
        {
            "id": "banco_financiero",
            "definicion": "Institución financiera que se dedica a administrar dinero, realizar préstamos, depósitos y otras operaciones monetarias",
            "ejemplos": [
                "Fui al banco a depositar mi sueldo",
                "El banco me aprobó el crédito hipotecario",
                "Necesito ir al banco a sacar dinero del cajero",
                "Hice una transacción en el banco",
                "Solicité un préstamo al banco"
            ],
            "sinonimos": ["entidad_bancaria", "institución_financiera", "caja", "entidad_crediticia"],
            "categoria": "sustantivo",
            "traduccion_en": "bank"
        },
        {
            "id": "banco_peces",
            "definicion": "Grupo numeroso de peces que nadan juntos de manera coordinada",
            "ejemplos": [
                "Vimos un banco de peces nadando cerca de la costa",
                "El banco de atunes se movía rápidamente"
            ],
            "sinonimos": ["cardumen", "grupo_de_peces"],
            "categoria": "sustantivo",
            "traduccion_en": "school"
        },
        {
            "id": "banco_trabajo",
            "definicion": "Mesa o superficie de trabajo resistente usada en talleres, laboratorios o cocinas",
            "ejemplos": [
                "El carpintero cortó la madera sobre el banco de trabajo",
                "Necesito limpiar el banco del taller"
            ],
            "sinonimos": ["mesa_de_trabajo", "mesón"],
            "categoria": "sustantivo",
            "traduccion_en": "workbench"
        }
    ]
}

# Clase para manejar sentidos
class SentidoPersonalizado:
    """Representa un sentido/significado de una palabra"""

    def __init__(self, sentido_dict):
        self._id = sentido_dict['id']
        self._definicion = sentido_dict['definicion']
        self._ejemplos = sentido_dict.get('ejemplos', [])
        self._sinonimos = sentido_dict.get('sinonimos', [])
        self._categoria = sentido_dict.get('categoria', '')
        self._traduccion_en = sentido_dict.get('traduccion_en', '')

    @property
    def id(self):
        return self._id

    def definition(self):
        return self._definicion

    def examples(self):
        return self._ejemplos

    def traduccion_ingles(self):
        return self._traduccion_en

    def __repr__(self):
        return f"Sentido({self._id})"


# Clase diccionario
class DiccionarioPersonalizado:
    """Diccionario personalizado para desambiguación"""

    def __init__(self, diccionario_data):
        self.diccionario = diccionario_data

    def synsets(self, palabra):
        """Obtener todos los sentidos de una palabra"""
        palabra_lower = palabra.lower()
        if palabra_lower not in self.diccionario:
            return []

        sentidos = []
        for sentido_dict in self.diccionario[palabra_lower]:
            sentidos.append(SentidoPersonalizado(sentido_dict))

        return sentidos


# Crear instancia del diccionario
DICCIONARIO = DiccionarioPersonalizado(DICCIONARIO_ESPANOL)
print(f"✅ Diccionario cargado con {len(DICCIONARIO_ESPANOL)} palabras")

✅ Diccionario cargado con 1 palabras


<a id="b_3"></a>
#  BLOQUE 3: Función de Tokenización Mejorada

In [ ]:
def optimized_tokenize(document: str, word: str, preserve_freq: bool = False,
                       use_lemmatization: bool = True) -> Set[str]:
    """
    Tokeniza y preprocesa un documento con lematización.

    Mejoras sobre el código original:
    - Añade lematización (el original no la tenía)
    - Elimina stopwords de forma más inteligente
    - Filtra la palabra objetivo para evitar ruido

    Args:
        document: Texto a tokenizar
        word: Palabra objetivo (se excluirá del resultado)
        preserve_freq: Si True, devuelve Counter; si False, devuelve set
        use_lemmatization: Si True, aplica lematización con Spacy

    Returns:
        Set de tokens procesados (o Counter si preserve_freq=True)
    """
    if not document or not isinstance(document, str):
        return Counter() if preserve_freq else set()

    # Tokenización básica
    tokenizer = nltk.RegexpTokenizer(r'\w+')
    tokens = tokenizer.tokenize(document)

    processed_tokens = []

    if use_lemmatization and NLP_ES:
        # Lematización con Spacy
        doc = NLP_ES(" ".join(tokens))
        for token in doc:
            t = token.lemma_.lower()
            # Filtrar: stopwords, no alfabéticos, palabra objetivo
            if (t not in STOPWORDS_ES and
                token.text.isalpha() and
                t != word.lower()):
                processed_tokens.append(t)
    else:
        # Sin lematización (fallback)
        for token in tokens:
            t = token.lower()
            if (t not in STOPWORDS_ES and
                t.isalpha() and
                t != word.lower()):
                processed_tokens.append(t)

    return Counter(processed_tokens) if preserve_freq else set(processed_tokens)


print("✅ Función de tokenización lista")

✅ Función de tokenización lista


<a id="b_4"></a>
#  BLOQUE 4: Algoritmo Weighted Lesk Mejorado

In [ ]:
def weighted_lesk_mejorado(document: str, word: str,
                          diccionario=None) -> Tuple[SentidoPersonalizado, List[float]]:
    """
    Algoritmo Weighted Lesk con mejoras sobre el original.

    Mejoras implementadas:
    1. Usa lematización (el original no)
    2. Calcula IDF correctamente para ponderar tokens raros
    3. Maneja casos edge correctamente (sin definiciones, sin contexto)
    4. Pre-tokeniza todas las definiciones para eficiencia

    Args:
        document: Contexto donde aparece la palabra
        word: Palabra ambigua a desambiguar
        diccionario: Diccionario a usar (por defecto DICCIONARIO)

    Returns:
        Tupla de (sentido_elegido, lista_de_pesos)
    """
    if diccionario is None:
        diccionario = DICCIONARIO

    # Obtener todos los sentidos posibles
    synsets = diccionario.synsets(word)

    if not synsets:
        print(f"⚠️ No se encontraron sentidos para '{word}'")
        return None, []

    # 1. Tokenizar el contexto
    context_tokens: Set[str] = optimized_tokenize(document, word, preserve_freq=False)

    if not context_tokens:
        # Sin contexto significativo, devolver primer sentido
        print("⚠️ Contexto vacío después de preprocesamiento")
        return synsets[0], [0] * len(synsets)

    N_t = len(synsets)  # Número total de sentidos

    # 2. Calcular Document Frequency (DF) para cada token
    # DF = en cuántos sentidos aparece cada token del contexto
    DF_map: Dict[str, int] = {token: 0 for token in context_tokens}
    sense_token_cache: List[Set[str]] = []

    # Pre-tokenizar todas las definiciones y ejemplos
    for sense in synsets:
        comparison_set = set()

        # Tokenizar definición
        definition = sense.definition()
        if definition:
            comparison_set = optimized_tokenize(definition, word, preserve_freq=False)

        # Tokenizar ejemplos
        examples = sense.examples()
        if examples:
            for example in examples:
                if example:
                    comparison_set.update(optimized_tokenize(example, word, preserve_freq=False))

        sense_token_cache.append(comparison_set)

        # Actualizar DF
        for token in context_tokens:
            if token in comparison_set:
                DF_map[token] += 1

    # 3. Calcular IDF (Inverse Document Frequency)
    # IDF = log(N_t / (DF + 1))
    # Tokens raros (bajo DF) tienen mayor peso
    IDF_map: Dict[str, float] = {}
    for token, df in DF_map.items():
        idf_value = np.log(N_t / (df + 1))
        IDF_map[token] = idf_value

    # 4. Calcular peso de cada sentido (solapamiento ponderado)
    weights: List[float] = [0.0] * N_t

    for index, comparison_set in enumerate(sense_token_cache):
        peso_total = 0.0

        # Sumar IDF de cada token que solapa
        for token in context_tokens:
            if token in comparison_set:
                peso_total += IDF_map[token]

        weights[index] = peso_total

    # 5. Seleccionar sentido con mayor peso
    max_weight = max(weights) if weights else 0.0

    if max_weight > 0.0:
        index = weights.index(max_weight)
    else:
        # Sin solapamiento, usar primer sentido
        print("⚠️ Sin solapamiento, usando primer sentido por defecto")
        index = 0

    return synsets[index], weights


print("✅ Algoritmo Weighted Lesk listo")

✅ Algoritmo Weighted Lesk listo


<a id="b_4"></a>
#  BLOQUE 5: Función de Traducción con Desambiguación

In [ ]:
def traducir_con_desambiguacion(texto: str, palabra_ambigua: str,
                                diccionario=None, translator=None,
                                verbose: bool = True) -> Dict:
    """
    Función principal: Desambigua y traduce usando Lesk + Transformer.

    Proceso:
    1. Desambigua la palabra con Weighted Lesk
    2. Obtiene la traducción correcta del sentido identificado
    3. Reemplaza la palabra ambigua con su traducción
    4. Traduce el texto completo con el modelo transformer
    5. Compara con traducción directa (sin desambiguación)

    Args:
        texto: Texto en español a traducir
        palabra_ambigua: Palabra que necesita desambiguación
        diccionario: Diccionario a usar
        translator: Pipeline de traducción
        verbose: Si True, imprime información detallada

    Returns:
        Dict con resultados y métricas
    """
    if diccionario is None:
        diccionario = DICCIONARIO
    if translator is None:
        translator = TRANSLATOR

    resultado = {
        'texto_original': texto,
        'palabra_ambigua': palabra_ambigua,
        'sentido_detectado': None,
        'traduccion_palabra': None,
        'traduccion_con_lesk': None,
        'traduccion_directa': None,
        'pesos_lesk': None,
        'tokens_contexto': None
    }

    if verbose:
        print("=" * 80)
        print("🔍 DESAMBIGUACIÓN Y TRADUCCIÓN CON LESK")
        print("=" * 80)
        print(f"\n📝 Texto original: '{texto}'")
        print(f"🎯 Palabra a desambiguar: '{palabra_ambigua}'")

    # 1. DESAMBIGUACIÓN CON LESK
    sentido, pesos = weighted_lesk_mejorado(texto, palabra_ambigua, diccionario)

    if sentido is None:
        print(f"\n❌ No se pudo desambiguar '{palabra_ambigua}'")
        return resultado

    resultado['sentido_detectado'] = sentido.id
    resultado['pesos_lesk'] = pesos
    resultado['tokens_contexto'] = list(optimized_tokenize(texto, palabra_ambigua))

    if verbose:
        print(f"\n✅ Sentido detectado: {sentido.id}")
        print(f"   Definición: {sentido.definition()}")
        print(f"   Pesos: {[round(p, 3) for p in pesos]}")

    # 2. OBTENER TRADUCCIÓN DEL SENTIDO
    traduccion_correcta = sentido.traduccion_ingles()
    resultado['traduccion_palabra'] = traduccion_correcta

    if verbose:
        print(f"\n🔤 Traducción de '{palabra_ambigua}': '{traduccion_correcta}'")

    # 3. INYECTAR TRADUCCIÓN EN EL TEXTO
    # Reemplazar la palabra ambigua con su traducción
    texto_forzado = texto.lower().replace(
        palabra_ambigua.lower(),
        traduccion_correcta
    )

    if verbose:
        print(f"\n💉 Texto con inyección: '{texto_forzado}'")

    # 4. TRADUCIR CON MODELO TRANSFORMER (con inyección)
    if translator:
        traduccion_forzada = translator(texto_forzado, max_length=100)[0]['translation_text']
        resultado['traduccion_con_lesk'] = traduccion_forzada

        if verbose:
            print(f"\n🌐 Traducción con Lesk: '{traduccion_forzada}'")

        # 5. TRADUCCIÓN DIRECTA (sin desambiguación)
        traduccion_pura = translator(texto, max_length=100)[0]['translation_text']
        resultado['traduccion_directa'] = traduccion_pura

        if verbose:
            print(f"🌐 Traducción directa: '{traduccion_pura}'")

            # Comparación
            print(f"\n📊 COMPARACIÓN:")
            palabra_en_lesk = traduccion_correcta in traduccion_forzada.lower()
            print(f"   ¿'{traduccion_correcta}' en traducción Lesk? {'✅ SÍ' if palabra_en_lesk else '❌ NO'}")
            print(f"   ¿Traducciones diferentes? {'✅ SÍ' if traduccion_forzada != traduccion_pura else '❌ NO (iguales)'}")

    if verbose:
        print("=" * 80)

    return resultado


print("✅ Función de traducción con desambiguación lista")

✅ Función de traducción con desambiguación lista


<a id="b_6"></a>
#  BLOQUE 6: Ejecución del Encargo

In [ ]:
# ============================================================================
# EJECUCIÓN DEL ENCARGO ESPECÍFICO
# ============================================================================

print("\n" + "🎯" * 40)
print("ENCARGO: Desambiguar 'banco' en contexto financiero")
print("🎯" * 40 + "\n")

# Frase del encargo
texto_encargo = "Fui por dinero, un préstamo e hice una transacción en el banco"
palabra_objetivo = "banco"

# Ejecutar desambiguación y traducción
resultado_encargo = traducir_con_desambiguacion(
    texto=texto_encargo,
    palabra_ambigua=palabra_objetivo,
    verbose=True
)

# ============================================================================
# PRUEBAS ADICIONALES CON OTROS CONTEXTOS
# ============================================================================

print("\n\n" + "🧪" * 40)
print("PRUEBAS ADICIONALES")
print("🧪" * 40 + "\n")

contextos_prueba = [
    "Me senté en el banco del parque",
    "El banco me aprobó el crédito",
    "Vimos un banco de peces en el mar"
]

for i, contexto in enumerate(contextos_prueba, 1):
    print(f"\n{'='*80}")
    print(f"PRUEBA {i}")
    print(f"{'='*80}")

    resultado = traducir_con_desambiguacion(
        texto=contexto,
        palabra_ambigua="banco",
        verbose=True
    )
    print()


🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯
ENCARGO: Desambiguar 'banco' en contexto financiero
🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯

🔍 DESAMBIGUACIÓN Y TRADUCCIÓN CON LESK

📝 Texto original: 'Fui por dinero, un préstamo e hice una transacción en el banco'
🎯 Palabra a desambiguar: 'banco'

✅ Sentido detectado: banco_financiero
   Definición: Institución financiera que se dedica a administrar dinero, realizar préstamos, depósitos y otras operaciones monetarias
   Pesos: [0.0, 3.466, 0.0, 0.0]

🔤 Traducción de 'banco': 'bank'

💉 Texto con inyección: 'fui por dinero, un préstamo e hice una transacción en el bank'

🌐 Traducción con Lesk: 'I went for money, a loan, and I made a transaction at the bank.'
🌐 Traducción directa: 'I went for money, a loan, and I made a transaction at the bank.'

📊 COMPARACIÓN:
   ¿'bank' en traducción Lesk? ✅ SÍ
   ¿Traducciones diferentes? ❌ NO (iguales)


🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪
PRUEBAS ADICIONALES
🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪🧪




<a id="b_7"></a>
# 📊 BLOQUE 7: Análisis Comparativo

In [ ]:
def analisis_comparativo_detallado(resultado: Dict):
    """
    Realiza un análisis detallado del resultado de la desambiguación.
    """
    print("\n" + "📊" * 40)
    print("ANÁLISIS DETALLADO")
    print("📊" * 40)

    print(f"\n🔍 INFORMACIÓN DEL CONTEXTO:")
    print(f"   Texto: '{resultado['texto_original']}'")
    print(f"   Tokens significativos: {resultado['tokens_contexto']}")
    print(f"   Cantidad de tokens: {len(resultado['tokens_contexto'])}")

    print(f"\n🎯 DESAMBIGUACIÓN:")
    print(f"   Sentido detectado: {resultado['sentido_detectado']}")
    print(f"   Traducción elegida: '{resultado['traduccion_palabra']}'")
    print(f"   Pesos Lesk: {[round(p, 3) for p in resultado['pesos_lesk']]}")

    print(f"\n🌐 TRADUCCIONES:")
    print(f"   Con Lesk: '{resultado['traduccion_con_lesk']}'")
    print(f"   Directa:  '{resultado['traduccion_directa']}'")

    # Verificar si la traducción es correcta
    palabra_correcta_presente = (
        resultado['traduccion_palabra'] in resultado['traduccion_con_lesk'].lower()
    )

    print(f"\n✅ VERIFICACIÓN:")
    if palabra_correcta_presente:
        print(f"   ✓ La palabra '{resultado['traduccion_palabra']}' está en la traducción")
        print(f"   ✓ La desambiguación fue EXITOSA")
    else:
        print(f"   ✗ La palabra '{resultado['traduccion_palabra']}' NO está en la traducción")
        print(f"   ✗ Revisar el proceso")

    # Comparar ambas traducciones
    if resultado['traduccion_con_lesk'] != resultado['traduccion_directa']:
        print(f"\n🔄 Las traducciones son DIFERENTES (Lesk tuvo efecto)")
    else:
        print(f"\n🔄 Las traducciones son IGUALES (el modelo ya desambiguó correctamente)")

    print("📊" * 40)


# Ejecutar análisis del resultado del encargo
analisis_comparativo_detallado(resultado_encargo)


📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊
ANÁLISIS DETALLADO
📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊

🔍 INFORMACIÓN DEL CONTEXTO:
   Texto: 'Fui por dinero, un préstamo e hice una transacción en el banco'
   Tokens significativos: ['hacer', 'préstamo', 'dinero', 'ir', 'transacción']
   Cantidad de tokens: 5

🎯 DESAMBIGUACIÓN:
   Sentido detectado: banco_financiero
   Traducción elegida: 'bank'
   Pesos Lesk: [0.0, 3.466, 0.0, 0.0]

🌐 TRADUCCIONES:
   Con Lesk: 'I went for money, a loan, and I made a transaction at the bank.'
   Directa:  'I went for money, a loan, and I made a transaction at the bank.'

✅ VERIFICACIÓN:
   ✓ La palabra 'bank' está en la traducción
   ✓ La desambiguación fue EXITOSA

🔄 Las traducciones son IGUALES (el modelo ya desambiguó correctamente)
📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊


<a id="conclusiones"></a>
# 🎓 Conclusiones y Aprendizajes: Desambiguación Léxica con Lesk




## 🔍 Aprendizajes

### 1. La Calidad del Diccionario es FUNDAMENTAL

**Descubrimiento más importante**: El método Lesk depende críticamente de la calidad de las definiciones y ejemplos.

#### Observaciones:

- ✅ **Con diccionario personalizado**: El algoritmo funciona excelentemente, identificando correctamente el sentido basándose en palabras clave como "dinero", "préstamo", "transacción".

- ❌ **Con WordNet español estándar**: Definiciones y ejemplos inexistentes, resultando en solapamientos mínimos o nulos.

- ⚠️ **Con la librería `wn` (MCR)**: A pesar de ser una mejora sobre WordNet estándar, las definiciones en español siguen siendo insuficientes para contextos específicos.

#### Lección:
> **Antes de implementar Lesk, invertir tiempo en conseguir o crear un buen diccionario es MÁS IMPORTANTE que optimizar el algoritmo.**

Para proyectos futuros con LESK:
- Diccionarios de dominio específico
- Combinar múltiples fuentes (Wiktionary, RAE, diccionarios técnicos)
- Crear diccionario personalizado para las 50-100 palabras más ambiguas del dominio

---

### 2. Los Modelos Neuronales Han Superado a Lesk para Traducción General

#### Mejor opción:
El modelo **Helsinki-NLP/opus-mt-es-en** desambigua correctamente la mayoría de casos SIN necesidad de Lesk:
- Traduce "banco" como "bank" en contextos financieros
- Traduce "banco" como "bench" en contextos de asientos
- Lo hace en una sola pasada, sin preprocesamiento

#### Conclusión:
> **Para traducción general, los modelos transformer han vuelto obsoleto el algoritmo Lesk. Sin embargo, Lesk sigue siendo válido para casos MUY específicos o dominios especializados.**

---

### 3. Mejoras Implementadas sobre el Código Original

El código original tenía varios problemas que fueron corregidos:

#### Problemas Identificados:

1. **❌ Sin lematización**:
   - Original: Comparaba "dinero" vs "dineros" como palabras diferentes
   - Mejorado: Lematiza ambas a "dinero"

2. **❌ Operador bitwise incorrecto**:
```python
   # Original (INCORRECTO)
   if not t.is_punct | t.is_stop:  # | es bitwise OR
   
   # Corregido
   if not (t.is_punct or t.is_stop):  # or es lógico